# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Print basic info
meta = dataset.metadata  # Keep the metadata object
meta_json = meta.to_json()
print(f"{meta_json['name']}: {meta_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we display all available record sets and their corresponding `@id` (unique Croissant identifiers), as well as their fields and field `@id`s.

In [ ]:
# List all record sets with their `@id`s
record_sets = list(dataset.metadata.record_sets())

if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '(no name)')}")
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - Field @id: {field['@id']} Name: {field.get('name', '')} DataType: {field.get('dataType', '')}")
        print('-'*40)

# For demo, print up to the first few records in each record set if available
for rs in record_sets:
    print(f"Example records from RecordSet @id: {rs['@id']}")
    try:
        for i, row in enumerate(dataset.records(record_set=rs['@id'])):
            pprint.pprint(row)
            if i >= 2:
                break
    except Exception as e:
        print(f"Could not read records: {e}")
    print('-'*40)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Replace `<record_set_id>` with one of the `@id` values from the printed record sets above.

In [ ]:
# To run the following cell, set a correct `record_set_id` from the overview above.
# For demonstration, we will attempt to use the first record set if possible.
record_sets = list(dataset.metadata.record_sets())
dataframes = {}

if not record_sets:
    print("No record sets defined in the schema.")
else:
    # Extract all record sets
    for rs in record_sets:
        rs_id = rs['@id']
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded dataframe for RecordSet @id: {rs_id}, shape: {df.shape}")
        else:
            print(f"RecordSet @id {rs_id} has no records.")

    # Print columns of the first record set as example
    first_id = record_sets[0]['@id']
    if first_id in dataframes:
        print(f"Columns in record set {first_id}:")
        print(dataframes[first_id].columns.tolist())
        display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

For demonstration, we use the first available record set and choose a numeric field (e.g., 'MW [kDa]' for Molecular Weight) by its field `@id`, if available.

In [ ]:
# Choose a record set and a numeric field for EDA
if not record_sets:
    print("No record sets available for EDA.")
else:
    rs = record_sets[0]
    rs_id = rs['@id']
    df = dataframes.get(rs_id, None)
    if df is None:
        print("No records loaded for the chosen record set.")
    else:
        # Try to pick a numeric field automatically if possible
        numeric_cols = df.select_dtypes(include='number').columns
        if len(numeric_cols) > 0:
            numeric_field = numeric_cols[0]
        else:
            # Try to convert a column with likely numeric values
            for col in df.columns:
                # Test conversion
                try:
                    df[col] = pd.to_numeric(df[col])
                    numeric_field = col
                    break
                except Exception:
                    continue
            else:
                numeric_field = None
        if numeric_field is None:
            print("Could not identify a numeric field in the record set.")
        else:
            threshold = df[numeric_field].mean() if not pd.isna(df[numeric_field].mean()) else 0
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold}:")
            print(filtered_df.head())

            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Group by a categorical field if available
            categorical_cols = df.select_dtypes(include='object').columns
            group_field = categorical_cols[0] if len(categorical_cols) > 0 else None

            if group_field:
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by {group_field}:")
                print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we show a histogram for the chosen numeric field and a boxplot grouped by the first categorical attribute if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_sets:
    print("No record sets to visualize.")
else:
    rs_id = record_sets[0]['@id']
    df = dataframes.get(rs_id, None)
    if df is not None and numeric_field is not None:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f"Histogram of {numeric_field} in {rs_id}")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()

        if group_field:
            # For display, take up to the first 10 unique category values
            unique_cats = df[group_field].dropna().unique()[:10]
            plt.figure(figsize=(10,4))
            sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(unique_cats)])
            plt.title(f"Boxplot of {numeric_field} grouped by {group_field}")
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded and explored the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library.
- We examined metadata, record sets, and fields (all by their Croissant `@id`s), loaded tabular data, and performed simple EDA and visualizations.
- This workflow can be extended for more advanced analyses, using the field and record set `@id`s to ensure reproducibility and schema-traceability.